In [1]:
import pandas as pd
import numpy as np
import gseapy as gp

from pathlib import Path

project_dir = Path.cwd().resolve().parent
results_dir = project_dir / "results"
figures_dir = project_dir / "figures"

figures_dir.mkdir(parents=True, exist_ok=True)

# Load the main comparison
limma_results = pd.read_csv(
    results_dir / "limma_Early_vs_Late.csv",
    index_col=0
)

print("Genes available:", len(limma_results))
display(limma_results.head())

Genes available: 730


,logFC,AveExpr,t,P.Value,adj.P.Val,B
ENG,1.246786,12.612691,5.484919,0.000005,0.003579,4.047988
CEACAM1,-0.861637,7.706398,-4.741072,0.000042,0.015496,2.101487
BATF,-0.843373,6.050905,-4.331198,0.000138,0.033538,1.038530
IDO1,-0.981739,10.880878,-4.219757,0.000189,0.034542,0.752453
IL12RB2,-0.770162,7.511955,-4.049974,0.000306,0.043251,0.319926


In [2]:
# Create ranked gene list
ranked_genes = (
    limma_results[["t"]]
    .dropna()
    .reset_index()
)

ranked_genes.columns = ["Gene", "Score"]

ranked_genes = ranked_genes.sort_values(
    "Score",
    ascending=False
)

print("Genes ranked:", len(ranked_genes))
display(ranked_genes.head(10))

Genes ranked: 730


,Gene,Score
0,ENG,5.484919
5,MME,3.996484
7,TREM1,3.815693
11,GATA3,3.669508
16,MAP2K4,3.414068
22,ITGA5,3.232117
25,LTF,3.198894
41,CSF2RB,2.983063
54,SMAD2,2.804569
80,PSEN1,2.622494


In [3]:
libraries = gp.get_library_name(organism="Human")

go_libraries = sorted([
    name for name in libraries
    if name.startswith("GO_Biological_Process_")
])

print("Available GO Biological Process libraries:")
print(go_libraries)

Available GO Biological Process libraries:
['GO_Biological_Process_2021', 'GO_Biological_Process_2023', 'GO_Biological_Process_2025', 'GO_Biological_Process_2026']


In [4]:
# Select the latest available GO Biological Process library
if not go_libraries:
    raise RuntimeError("No GO Biological Process library found")

gene_set_library = go_libraries[-1]
print("Using:", gene_set_library)

gsea_results = gp.prerank(
    rnk=ranked_genes,
    gene_sets=gene_set_library,
    min_size=5,
    max_size=500,
    permutation_num=1000,
    seed=42,
    outdir=str(results_dir / "GSEA_Early_vs_Late"),
    verbose=True
)

display(gsea_results.res2d.head(10))

2026-11-06 12:55:06,957 [INFO] Parsing data files for GSEA.............................


Using: GO_Biological_Process_2026


2026-11-06 12:55:07,408 [INFO] Downloading and generating Enrichr library gene sets......
2026-11-06 12:55:08,048 [INFO] 4833 gene_sets have been filtered out when max_size=500 and min_size=5
2026-11-06 12:55:08,048 [INFO] 1121 gene_sets used for further statistical testing.....
2026-11-06 12:55:08,049 [INFO] Start to run GSEA...Might take a while..................
2026-11-06 12:55:11,249 [INFO] Congratulations. GSEApy runs successfully................



,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Kidney Development (GO:0001822),0.948199,2.293413,0.000000,0.048664,0.037,4/5,3.01%,MME;GATA3;VEGFA;CTSH
1,prerank,Renal System Development (GO:0072001),0.834241,2.021100,0.000000,0.320187,0.360,3/5,1.78%,MME;GATA3;VEGFA
2,prerank,Heart Morphogenesis (GO:0003007),0.807088,1.928333,0.003610,0.418437,0.558,3/5,1.78%,ENG;SMAD2;VEGFA
3,prerank,Transforming Growth Factor Beta Receptor Super...,0.493490,1.882282,0.000000,0.423875,0.658,6/14,10.14%,ENG;SMAD2;PPARG;MAPK3;MAP3K7;SMAD3
4,prerank,Positive Regulation of Glial Cell Migration (G...,-0.874340,-1.811532,0.001337,0.946831,0.509,3/5,8.77%,CCL3;CX3CL1;CSF1
5,prerank,Interleukin-5-mediated Signaling Pathway (GO:0...,0.648291,1.810270,0.016807,0.521148,0.801,3/7,3.97%,CSF2RB;IL5RA;LYN
6,prerank,Regulation of Microglial Cell Migration (GO:19...,-0.866871,-1.793176,0.000000,0.604703,0.595,4/5,11.10%,CCL3;CX3CL1;CSF1;CX3CR1
7,prerank,Positive Regulation of Phosphorylation (GO:004...,0.551389,1.792676,0.021164,0.475581,0.827,10/10,45.62%,VEGFA;MIF;IL34;PLAUR;CDKN1A;CSF1R;C3;IRGM;THBS...
8,prerank,Regulation of Amyloid Fibril Formation (GO:190...,0.733455,1.778894,0.015810,0.438767,0.846,2/5,1.92%,PSEN1;CLU
9,prerank,Wound Healing (GO:0042060),0.698643,1.767914,0.008197,0.405903,0.856,3/6,10.14%,ENG;SLC11A1;SMAD3


In [5]:
# Repeat GSEA using larger gene sets
gsea_sensitivity = gp.prerank(
    rnk=ranked_genes,
    gene_sets=gene_set_library,
    min_size=10,
    max_size=500,
    permutation_num=5000,
    seed=42,
    outdir=str(results_dir / "GSEA_Sensitivity"),
    verbose=False
)

# Sort by FDR
sensitivity_results = (
    gsea_sensitivity.res2d
    .sort_values("FDR q-val")
)

print("Pathways tested:", len(sensitivity_results))

print(
    "Significant pathways (FDR < 0.05):",
    (sensitivity_results["FDR q-val"] < 0.05).sum()
)

display(
    sensitivity_results[
        ["Term", "NES", "NOM p-val", "FDR q-val"]
    ].head(10)
)

# Save results
sensitivity_results.to_csv(
    results_dir / "GSEA_sensitivity_results.csv",
    index=False
)

Pathways tested: 503
Significant pathways (FDR < 0.05): 0


,Term,NES,NOM p-val,FDR q-val
4,Regulation of Protein Phosphorylation (GO:0001...,1.710152,0.012136,0.218234
19,Transforming Growth Factor Beta Receptor Signa...,1.540519,0.041147,0.272420
1,Positive Regulation of Phosphorylation (GO:004...,1.790753,0.017186,0.274436
3,Regulation of Cell Adhesion (GO:0030155),1.719690,0.007109,0.276580
29,Positive Regulation of Leukocyte Proliferation...,1.496375,0.039634,0.304758
18,Intracellular Receptor Signaling Pathway (GO:0...,1.541155,0.043870,0.310487
39,Regulation of Gene Expression (GO:0010468),1.466195,0.000000,0.319513
15,Epithelial Cell Differentiation (GO:0030855),1.559223,0.046892,0.329023
0,Transforming Growth Factor Beta Receptor Super...,1.863670,0.003974,0.351132
14,Negative Regulation of Cell Motility (GO:2000146),1.561020,0.043742,0.391103
